# ProboSed — 05: MTD Physical Property Quantification

**IODP Expedition 405 — Site C0019J, Japan Trench**

Tests whether MTD intervals identified from VCD observations in notebook 01
show statistically distinct physical properties compared to the surrounding
stable matrix. This is the quantitative validation step connecting the
stability index to independent instrument measurements.

**Datasets used:**
- `C0019J_MTD_catalog.csv` — MTD boundaries from notebook 01 (pipeline output)
- `C0019_Resistivity.xlsx` — Electrical resistivity, 0.4-873.5 m CSF-A
- `C0019J_PWVD.xlsx` — P-wave velocity, ~100-870 m CSF-A
- `MAD_C0019J.csv` — Moisture and density (porosity), Hole J

**Scientific question:**
Do intervals classified as MTD from VCD fabric observations show lower Vp,
lower resistivity, and higher porosity than the surrounding stable matrix?
If yes, this confirms the stability index captures real mechanical contrasts
rather than artifacts of the VCD classification scheme.

**Output:**
- Statistical comparison table: MTD vs stable matrix for each property
- Multi-panel depth profile with MTD intervals shaded
- Box plots comparing physical properties by MTD classification
- `C0019J_MTD_properties.csv` — merged dataset for further analysis

In [ ]:
# Cell 1: Install dependencies
!pip install pandas openpyxl matplotlib scipy numpy --quiet

In [ ]:
# Cell 2: Mount Drive and clone ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

repo_url  = 'https://github.com/rocknrene/ProboSed.git'
repo_path = '/content/ProboSed'

if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

result = subprocess.run(['git', 'clone', repo_url, repo_path],
                        capture_output=True, text=True)
print('Return code:', result.returncode)
print('STDERR:', result.stderr)

sys.path.insert(0, repo_path)
print('ProboSed modules available')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Return code: 0
STDERR: Cloning into '/content/ProboSed'...

ProboSed modules available


In [ ]:
# Cell 3: Configure paths
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pipeline output from notebook 01
MTD_CSV = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')

# Physical property data files
RES_FILE  = os.path.join(DATA_DIR, 'C0019_Resistivity.xlsx')
VP_FILE   = os.path.join(DATA_DIR, 'C0019J_PWVD.xlsx')
MAD_FILE  = os.path.join(DATA_DIR, 'MAD_C0019J.csv')

# Core summary Excel — same file used in notebook 01 to build the depth backbone.
# Required here to assign CSF-A depths to MAD samples, which have no
# depth column and must be matched by core/section label.
SUMMARY_XLSX = os.path.join(DATA_DIR, '405-C0019J_CoreSummary_2410280924.xlsx')

# Verify all files exist
for path, label in [
    (MTD_CSV,  'MTD catalog (notebook 01 output)'),
    (RES_FILE, 'Resistivity'),
    (VP_FILE,  'P-wave velocity'),
    (MAD_FILE, 'MAD porosity'),
]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check path'
    print(f'{label}: {status}')
    print(f'  {path}')

MTD catalog (notebook 01 output): FOUND
  /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_catalog.csv
Resistivity: FOUND
  /content/drive/MyDrive/iodp/X405/Data & Data Tracking/C0019_Resistivity.xlsx
P-wave velocity: FOUND
  /content/drive/MyDrive/iodp/X405/Data & Data Tracking/C0019J_PWVD.xlsx
MAD porosity: FOUND
  /content/drive/MyDrive/iodp/X405/Data & Data Tracking/MAD_C0019J.csv


## Step 1 — Load the MTD catalog

The MTD catalog is the output of notebook 01. It defines the depth boundaries
of each MTD interval identified from VCD fabric observations. These boundaries
are used to classify every physical property measurement as either
**within an MTD** or **stable matrix** before any physical property data
is examined — preventing circular reasoning.

In [ ]:
# Cell 4: Load MTD catalog from notebook 01
import pandas as pd
import numpy as np

mtd_catalog = pd.read_csv(MTD_CSV)
print(f'MTD intervals loaded: {len(mtd_catalog)}')
print(f'Total MTD thickness: {mtd_catalog["thickness_m"].sum():.1f} m')
print()
print(mtd_catalog.to_string(index=False))

MTD intervals loaded: 10
Total MTD thickness: 111.0 m

mtd_id  top_m  bottom_m  thickness_m  mean_stability
 MTD-1   74.0      76.5          2.5             0.0
 MTD-2  167.5     177.0          9.5             1.0
 MTD-3  253.0     262.5          9.5             1.0
 MTD-4  272.0     291.0         19.0             0.0
 MTD-5  300.5     310.0          9.5             1.0
 MTD-6  406.0     415.5          9.5             0.0
 MTD-7  425.0     453.5         28.5             1.0
 MTD-8  606.5     616.0          9.5             1.0
 MTD-9  702.0     711.5          9.5             0.0
MTD-10  940.0     944.0          4.0             1.0


## Step 2 — Load and classify physical property data

Each physical property dataset is loaded and classified as MTD or stable matrix
using the VCD-derived boundaries. The classification is applied using a
nearest-depth lookup with a ±5 m tolerance to account for the spatial
resolution of VCD observations relative to instrument sampling intervals.

**Note on depth ranges:**
- Resistivity covers 0.4-873.5 m — matches MTD-1 through MTD-9
- Vp covers ~100-870 m — matches MTD-2 through MTD-9
- Porosity (MAD) covers the full hole — matches all MTDs with data

MTD-10 at 940-944 m is below the deepest instrument measurements
and is excluded from statistical comparisons but retained in the catalog.

In [ ]:
# Cell 5: Load resistivity data
res = pd.read_excel(RES_FILE)
res.columns = [c.strip() for c in res.columns]

# identify the depth and resistivity columns
depth_col = [c for c in res.columns if 'TopDepth' in c or 'Depth' in c][0]
res_y_col = [c for c in res.columns if 'ResistivityY' in c or 'Resistivity_Y' in c][0]

res = res[[depth_col, res_y_col]].dropna().rename(columns={
    depth_col: 'Depth_m',
    res_y_col: 'Resistivity_Ohm'
}).sort_values('Depth_m').reset_index(drop=True)

print(f'Resistivity: {len(res)} measurements, {res["Depth_m"].min():.1f} - {res["Depth_m"].max():.1f} m')
res.head()

Resistivity: 121 measurements, 130.3 - 873.5 m


,Depth_m,Resistivity_Ohm
0,130.300,3.687
1,167.990,0.901
2,179.805,1.084
3,188.195,1.046
4,190.765,3.406


In [ ]:
# Cell 6: Load P-wave velocity data
vp = pd.read_excel(VP_FILE)
vp.columns = [c.strip() for c in vp.columns]

depth_col = [c for c in vp.columns if 'TopDepth' in c or 'Depth' in c][0]
vp_col    = [c for c in vp.columns if 'VelocityX' in c or 'Vp_X' in c][0]

vp = vp[[depth_col, vp_col]].dropna().rename(columns={
    depth_col: 'Depth_m',
    vp_col:    'Vp_ms'
}).sort_values('Depth_m').reset_index(drop=True)

print(f'P-wave velocity: {len(vp)} measurements, {vp["Depth_m"].min():.1f} - {vp["Depth_m"].max():.1f} m')
vp.head()

P-wave velocity: 105 measurements, 130.3 - 823.3 m


,Depth_m,Vp_ms
0,130.300,1682.0
1,167.990,1679.0
2,179.805,1686.0
3,188.195,1650.0
4,190.765,1693.0


In [ ]:
# Cell 7: Load MAD porosity with depth from core summary
# MAD file has no depth column — depth is extracted from the sample
# label (e.g. '1K1', '2K3') matched to the core summary backbone

mad_raw = pd.read_csv(MAD_FILE, skiprows=2)
mad_raw.columns = [c.strip().lower() for c in mad_raw.columns]

# find porosity column
por_col = [c for c in mad_raw.columns if 'porosity' in c.lower()][0]
print(f'Porosity column: {por_col}')

# extract core number and section from comment e.g. '1K1' -> core=1, section=1
mad_raw['core_id'] = mad_raw['comment on measurement'].str.extract(r'^(\d+[A-Z])', expand=False)
mad_raw['section'] = mad_raw['comment on measurement'].str.extract(r'^\d+[A-Z](\d+)', expand=False)

# load depth backbone from core summary (same file notebook 01 uses)
summary_raw = pd.read_excel(SUMMARY_XLSX, header=5)
summary_raw.columns = [c.replace('\n', ' ').strip() for c in summary_raw.columns]
summary_raw = summary_raw.dropna(subset=['Core curated'])
summary_raw['core_id'] = summary_raw['Core curated'].astype(str).apply(
    lambda x: x.split('-')[-1].upper() if '-' in x else x.upper()
)

# build core top depth lookup
core_depths = summary_raw[['core_id', 'Top depth [m CSF-A]']].dropna()
core_depths = core_depths.rename(columns={'Top depth [m CSF-A]': 'core_top_m'})

# merge MAD with core depths, interpolate section depth
mad_merged = mad_raw.merge(core_depths, on='core_id', how='left')
mad_merged['section'] = pd.to_numeric(mad_merged['section'], errors='coerce')
mad_merged['Depth_m'] = mad_merged['core_top_m'] + (mad_merged['section'] - 1) * 1.5

mad = mad_merged[['Depth_m', por_col]].dropna().rename(
    columns={por_col: 'Porosity_vv'}
).sort_values('Depth_m').reset_index(drop=True)

print(f'MAD porosity: {len(mad)} measurements, {mad["Depth_m"].min():.1f} - {mad["Depth_m"].max():.1f} m')
mad.head()

Porosity column: porosity []
MAD porosity: 263 measurements, 82.0 - 828.0 m


,Depth_m,Porosity_vv
0,82.0,0.5711
1,83.5,0.6116
2,85.0,0.5234
3,91.5,0.5901
4,93.0,0.5757


## Step 3 — Classify measurements as MTD or stable matrix

For each measurement, check whether its depth falls within any MTD interval
from the catalog. Measurements within an MTD interval are labeled with the
MTD ID; all others are labeled as stable matrix.

This classification uses the pipeline-derived boundaries — not hardcoded
depth ranges — ensuring it updates automatically if the MTD catalog changes.

In [ ]:
# Cell 8: Classification function
def classify_by_mtd(df, mtd_catalog, depth_col='Depth_m', tolerance=5.0):
    """
    Classify each measurement as MTD or stable matrix using the
    pipeline-derived MTD catalog boundaries.

    Parameters
    ----------
    df         : pd.DataFrame with a depth column
    mtd_catalog: pd.DataFrame from notebook 01 (top_m, bottom_m, mtd_id)
    depth_col  : name of the depth column in df
    tolerance  : extend each MTD boundary by this many meters (m)
                 accounts for spatial resolution of VCD observations

    Returns
    -------
    pd.DataFrame with added columns: Classification, MTD_ID
    """
    df = df.copy()
    df['Classification'] = 'Stable matrix'
    df['MTD_ID']         = None

    for _, row in mtd_catalog.iterrows():
        # extend boundaries by tolerance to account for VCD resolution
        top    = row['top_m']    - tolerance
        bottom = row['bottom_m'] + tolerance
        mask   = (df[depth_col] >= top) & (df[depth_col] <= bottom)
        df.loc[mask, 'Classification'] = 'MTD interval'
        df.loc[mask, 'MTD_ID']         = row['mtd_id']

    return df

# Apply classification to all three datasets
res_c = classify_by_mtd(res, mtd_catalog)
vp_c  = classify_by_mtd(vp,  mtd_catalog)
mad_c = classify_by_mtd(mad, mtd_catalog)

print('Resistivity classification:')
print(res_c['Classification'].value_counts())
print()
print('Vp classification:')
print(vp_c['Classification'].value_counts())
print()
print('Porosity classification:')
print(mad_c['Classification'].value_counts())

Resistivity classification:
Classification
Stable matrix    92
MTD interval     29
Name: count, dtype: int64

Vp classification:
Classification
Stable matrix    76
MTD interval     29
Name: count, dtype: int64

Porosity classification:
Classification
Stable matrix    196
MTD interval      67
Name: count, dtype: int64


## Step 4 — Statistical comparison

Compare physical properties inside vs outside MTD intervals.
A Mann-Whitney U test is used rather than a t-test because the
distributions may not be normally distributed and sample sizes
within individual MTD intervals are small.

In [ ]:
# Cell 9: Statistical comparison
from scipy import stats

print('=== Physical Property Comparison: MTD vs Stable Matrix ===')
print()

datasets = [
    ('Resistivity (Ohm-m)', res_c,  'Resistivity_Ohm',
     'Lower resistivity in MTD expected: higher porosity/fluid content'),
    ('P-wave velocity (m/s)', vp_c, 'Vp_ms',
     'Lower Vp in MTD expected: weaker, less consolidated material'),
    ('Porosity (v/v)', mad_c, 'Porosity_vv',
     'Higher porosity in MTD expected: disaggregated, underconsolidated'),
]

results = []
for name, df, col, interpretation in datasets:
    mtd_vals    = df[df['Classification'] == 'MTD interval'][col].dropna()
    matrix_vals = df[df['Classification'] == 'Stable matrix'][col].dropna()

    if len(mtd_vals) < 3:
        print(f'{name}: insufficient MTD measurements (n={len(mtd_vals)}) — skip')
        continue

    # Mann-Whitney U test (non-parametric)
    stat, pval = stats.mannwhitneyu(mtd_vals, matrix_vals, alternative='two-sided')

    result = {
        'Property':         name,
        'MTD_n':            len(mtd_vals),
        'MTD_mean':         mtd_vals.mean(),
        'MTD_median':       mtd_vals.median(),
        'Matrix_n':         len(matrix_vals),
        'Matrix_mean':      matrix_vals.mean(),
        'Matrix_median':    matrix_vals.median(),
        'Mann_Whitney_p':   pval,
        'Significant':      pval < 0.05,
    }
    results.append(result)

    direction = 'LOWER' if mtd_vals.mean() < matrix_vals.mean() else 'HIGHER'
    print(f'{name}')
    print(f'  MTD:    mean={mtd_vals.mean():.3f}  median={mtd_vals.median():.3f}  n={len(mtd_vals)}')
    print(f'  Matrix: mean={matrix_vals.mean():.3f}  median={matrix_vals.median():.3f}  n={len(matrix_vals)}')
    print(f'  Mann-Whitney p = {pval:.4f}  -> {"SIGNIFICANT" if pval < 0.05 else "not significant"} (p<0.05)')
    print(f'  Direction: MTD is {direction} than stable matrix')
    print(f'  Interpretation: {interpretation}')
    print()

stats_df = pd.DataFrame(results)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'C0019J_MTD_statistics.csv'), index=False)
print(f'Statistics saved to: {OUTPUT_DIR}/C0019J_MTD_statistics.csv')

=== Physical Property Comparison: MTD vs Stable Matrix ===

Resistivity (Ohm-m)
  MTD:    mean=1.398  median=1.412  n=29
  Matrix: mean=20.476  median=1.520  n=92
  Mann-Whitney p = 0.0304  -> SIGNIFICANT (p<0.05)
  Direction: MTD is LOWER than stable matrix
  Interpretation: Lower resistivity in MTD expected: higher porosity/fluid content

P-wave velocity (m/s)
  MTD:    mean=1775.586  median=1777.000  n=29
  Matrix: mean=1849.908  median=1838.500  n=76
  Mann-Whitney p = 0.0017  -> SIGNIFICANT (p<0.05)
  Direction: MTD is LOWER than stable matrix
  Interpretation: Lower Vp in MTD expected: weaker, less consolidated material

Porosity (v/v)
  MTD:    mean=0.525  median=0.537  n=67
  Matrix: mean=0.511  median=0.508  n=196
  Mann-Whitney p = 0.0132  -> SIGNIFICANT (p<0.05)
  Direction: MTD is HIGHER than stable matrix
  Interpretation: Higher porosity in MTD expected: disaggregated, underconsolidated

Statistics saved to: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_stat

## Step 5 — Figures

### Figure A — Depth profiles without lithological column

Three-panel depth profile showing resistivity (log scale), P-wave velocity, and porosity
with MTD intervals shaded pink. Points are colored red for measurements within MTD
intervals and blue for stable matrix. MTD labels are annotated on the left edge of each
panel. This is the primary visual validation figure — if red points consistently cluster
at lower resistivity, lower Vp, and higher porosity than surrounding blue points, the
convergent validation argument is confirmed.

### Figure B — Depth profiles with lithological column

Same three physical property panels preceded by the Exp. 405 C0019J lithological column.
The column shows lithostratigraphic units 1 through 7 with fault zone positions marked.
This version connects the MTD intervals directly to the published stratigraphic framework.

In [ ]:
# Cell 10A: Figure A — 3-panel WITHOUT lithological column
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

C_MTD     = '#d62728'
C_MATRIX  = '#4575b4'
C_SHADE   = '#ffcccc'
C_FAULT   = '#cc0000'
DEPTH_MAX = 880
ALPHA_SH  = 0.35
PT_SIZE   = 40

def add_mtd_labels(ax, mtd_catalog, depth_max):
    for _, row in mtd_catalog.iterrows():
        mid = (row['top_m'] + row['bottom_m']) / 2
        if mid > depth_max:
            continue
        ax.annotate(row['mtd_id'], xy=(0.02, mid),
                    xycoords=('axes fraction', 'data'),
                    fontsize=7, color='darkred', fontweight='bold', va='center')

def setup_panel(ax, mtd_catalog, depth_max):
    ax.set_facecolor('white')
    ax.invert_yaxis()
    ax.set_ylim(depth_max, 0)
    ax.grid(axis='x', alpha=0.25, color='lightgray')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for _, row in mtd_catalog.iterrows():
        if row['top_m'] > depth_max:
            continue
        ax.axhspan(row['top_m'], min(row['bottom_m'], depth_max),
                   alpha=ALPHA_SH, color=C_SHADE, zorder=1)
    for fd in [210, 610, 845]:
        if fd <= depth_max:
            ax.axhline(fd, color=C_FAULT, ls='--', lw=1.2, alpha=0.8, zorder=2)

def pt_colors(df):
    return [C_MTD if c == 'MTD interval' else C_MATRIX for c in df['Classification']]

fig, axes = plt.subplots(1, 3, figsize=(12, 14), sharey=True)
fig.patch.set_facecolor('white')
plt.subplots_adjust(wspace=0.08)

# Panel 1: Resistivity (log scale)
setup_panel(axes[0], mtd_catalog, DEPTH_MAX)
axes[0].scatter(res_c['Resistivity_Ohm'], res_c['Depth_m'],
                c=pt_colors(res_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
axes[0].set_xscale('log')
axes[0].set_xlabel('Resistivity (Ω·m) log scale', fontsize=10)
axes[0].set_ylabel('Depth (m CSF-A)', fontsize=11)
axes[0].set_title('Electrical\nResistivity', fontsize=11, fontweight='bold')
add_mtd_labels(axes[0], mtd_catalog, DEPTH_MAX)

# Panel 2: Vp
setup_panel(axes[1], mtd_catalog, DEPTH_MAX)
axes[1].scatter(vp_c['Vp_ms'], vp_c['Depth_m'],
                c=pt_colors(vp_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
axes[1].set_xlabel('Vp (m/s)', fontsize=10)
axes[1].set_title('P-wave\nVelocity', fontsize=11, fontweight='bold')

# Panel 3: Porosity
setup_panel(axes[2], mtd_catalog, DEPTH_MAX)
axes[2].scatter(mad_c['Porosity_vv'], mad_c['Depth_m'],
                c=pt_colors(mad_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
axes[2].set_xlabel('Porosity (v/v)', fontsize=10)
axes[2].set_title('Porosity\n(MAD)', fontsize=11, fontweight='bold')

legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_MTD, markersize=9, label='MTD interval', markeredgecolor='white'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_MATRIX, markersize=9, label='Stable matrix', markeredgecolor='white'),
    Line2D([0],[0], color=C_FAULT, ls='--', lw=1.5, label='Major fault zone'),
    mpatches.Patch(facecolor=C_SHADE, alpha=0.6, label=f'MTD zone (n={len(mtd_catalog)}, VCD-derived)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9,
           facecolor='white', bbox_to_anchor=(0.5, 0.01))
fig.suptitle('Site C0019J — Integrated Geotechnical & Stability Profile\nIODP Expedition 405 JTRACK',
             fontsize=12, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0, 0.05, 1, 0.97])

out_a = os.path.join(OUTPUT_DIR, 'C0019J_MTD_profile_3panel.png')
plt.savefig(out_a, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_a}')

Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_profile_3panel.png


In [ ]:
# Cell 10B: Figure B — 4-panel WITH lithological column
import matplotlib.gridspec as gridspec

lith_units = [
    (0,   82,  '1\nOlive-black\nsiliceous\nvitric mud',       '#4a4a4a'),
    (82,  160, '2A\nOlive-black\nsiliceous\nvitric mud',      '#3d3d3d'),
    (160, 210, '2B\nOlive-gray\nsiliceous\nvitric mud',       '#7a8a7a'),
    (210, 270, '3A\nOlive-black\nsiliceous\nvitric mud',      '#4a4a4a'),
    (270, 320, '3B\nGray siliceous\nvitric mud',              '#8a8a8a'),
    (320, 400, '4A\nOlive-black\nsiliceous\nvitric mud',      '#4a4a4a'),
    (400, 470, '4B\nGray vitric\nmud(-stone)',                '#8a8a8a'),
    (470, 530, '5A\nOlive-black\nsiliceous\nvitric mud',      '#4a4a4a'),
    (530, 570, '5B\nGray siliceous\nvitric mud',              '#8a8a8a'),
    (570, 610, '5C\nYellowish-brown\nsiliceous\nvitric mud',  '#8B6914'),
    (610, 760, '6A\nOlive-black\nsiliceous\nvitric mud',      '#4a4a4a'),
    (760, 845, '6B\nGreenish-gray\nvitric mud(-stone)',       '#5a7a5a'),
    (845, 880, '7\nColor-banded\nclay & chert',               '#c8a000'),
]
fault_depths = [210, 610, 845]
fault_labels = ['Thrust\nfault', 'Thrust\nfault', 'Plate boundary\nfault zone']

fig = plt.figure(figsize=(16, 14))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 2, 2, 2], wspace=0.06)

ax_lith = fig.add_subplot(gs[0])
ax_res  = fig.add_subplot(gs[1])
ax_vp   = fig.add_subplot(gs[2], sharey=ax_res)
ax_por  = fig.add_subplot(gs[3], sharey=ax_res)

# Litho column
ax_lith.set_facecolor('white')
ax_lith.set_xlim(0, 1)
ax_lith.set_xticks([])
ax_lith.invert_yaxis()
ax_lith.set_ylim(DEPTH_MAX, 0)
ax_lith.set_ylabel('Depth (m CSF-A)', fontsize=11)
ax_lith.set_title('Lith.\nUnit', fontsize=10, fontweight='bold')
ax_lith.spines['top'].set_visible(False)
ax_lith.spines['right'].set_visible(False)
ax_lith.spines['bottom'].set_visible(False)

for (top, bot, label, color) in lith_units:
    bot_c = min(bot, DEPTH_MAX)
    if top >= DEPTH_MAX:
        continue
    ax_lith.barh((top + bot_c)/2, 1, height=bot_c - top,
                 color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    if (bot_c - top) > 35:
        ax_lith.text(0.5, (top + bot_c)/2, label,
                     ha='center', va='center', fontsize=5,
                     color='white', fontweight='bold')

for fd, fl in zip(fault_depths, fault_labels):
    if fd <= DEPTH_MAX:
        ax_lith.axhline(fd, color=C_FAULT, ls='--', lw=1.2, alpha=0.8)
        ax_lith.annotate(fl, xy=(1.02, fd), xycoords=('axes fraction','data'),
                         fontsize=6, color=C_FAULT, va='center', style='italic')

# Data panels
for ax in [ax_res, ax_vp, ax_por]:
    setup_panel(ax, mtd_catalog, DEPTH_MAX)
    plt.setp(ax.get_yticklabels(), visible=False)

ax_res.scatter(res_c['Resistivity_Ohm'], res_c['Depth_m'],
               c=pt_colors(res_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
ax_res.set_xscale('log')
ax_res.set_xlabel('Resistivity\n(Ω·m) log scale', fontsize=9)
ax_res.set_title('Electrical\nResistivity', fontsize=11, fontweight='bold')
add_mtd_labels(ax_res, mtd_catalog, DEPTH_MAX)

ax_vp.scatter(vp_c['Vp_ms'], vp_c['Depth_m'],
              c=pt_colors(vp_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
ax_vp.set_xlabel('Vp (m/s)', fontsize=9)
ax_vp.set_title('P-wave\nVelocity', fontsize=11, fontweight='bold')

ax_por.scatter(mad_c['Porosity_vv'], mad_c['Depth_m'],
               c=pt_colors(mad_c), s=PT_SIZE, alpha=0.85, edgecolors='white', lw=0.3, zorder=3)
ax_por.set_xlabel('Porosity (v/v)', fontsize=9)
ax_por.set_title('Porosity\n(MAD)', fontsize=11, fontweight='bold')

legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_MTD, markersize=9, label='MTD interval', markeredgecolor='white'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_MATRIX, markersize=9, label='Stable matrix', markeredgecolor='white'),
    Line2D([0],[0], color=C_FAULT, ls='--', lw=1.5, label='Major fault zone'),
    mpatches.Patch(facecolor=C_SHADE, alpha=0.6, label=f'MTD zone (±5m, VCD-derived, n={len(mtd_catalog)})'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9,
           facecolor='white', bbox_to_anchor=(0.5, 0.005))
fig.suptitle('Site C0019J — MTD Stability Zones vs. Exp. 405 Stratigraphy\nIODP Expedition 405 JTRACK',
             fontsize=12, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0.05, 1, 0.97])

out_b = os.path.join(OUTPUT_DIR, 'C0019J_MTD_profile_4panel.png')
plt.savefig(out_b, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_b}')

/tmp/ipykernel_14573/618736852.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.05, 1, 0.97])


Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_profile_4panel.png


### Figure 2 — Box plots: MTD vs stable matrix

Box plots show the full distribution of each physical property inside
and outside MTD intervals. This is the clearest way to show whether the
difference between groups is statistically and scientifically meaningful.

In [ ]:
#import matplotlib.pyplot as plt
import numpy as np

# Updated Color Palette to match the 4-panel figure
C_MTD_RED = '#d62728'     # Matching the "MTD interval" red
C_STABLE_BLUE = '#4c72b0' # Matching the "Stable matrix" blue

fig, axes = plt.subplots(1, 3, figsize=(14, 6))
fig.patch.set_facecolor('white')

plot_data = [
    ('Resistivity (Ω·m)',   res_c,  'Resistivity_Ohm'),
    ('P-wave velocity (m/s)', vp_c, 'Vp_ms'),
    ('Porosity (v/v)',     mad_c,  'Porosity_vv'),
]

for ax, (name, df, col) in zip(axes, plot_data):
    ax.set_facecolor('white')

    # Group data
    mtd_data = df[df['Classification'] == 'MTD interval'][col].dropna().values
    stable_data = df[df['Classification'] == 'Stable matrix'][col].dropna().values

    groups = [mtd_data, stable_data]
    labels = [f'MTD interval\n(n={len(mtd_data)})',
              f'Stable matrix\n(n={len(stable_data)})']

    # Create Boxplot
    bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                medianprops=dict(color='black', lw=2),
                widths=0.6,
                showfliers=False)

    # Apply matching colors
    # Box 1: MTD
    bp['boxes'][0].set_facecolor(C_MTD_RED)
    bp['boxes'][0].set_alpha(0.8)

    # Box 2: Stable Matrix
    bp['boxes'][1].set_facecolor(C_STABLE_BLUE)
    bp['boxes'][1].set_alpha(0.8)

    # Add individual points (matching the scatter colors exactly)
    # i=0 for MTD (red), i=1 for Stable (blue)
    point_colors = [C_MTD_RED, C_STABLE_BLUE]
    for i, (grp, color) in enumerate(zip(groups, point_colors)):
        x = np.random.normal(i + 1, 0.04, size=len(grp))
        ax.scatter(x, grp, alpha=0.5, s=20, color=color, zorder=3, edgecolors='white', linewidth=0.5)

    # Add p-value annotation
    row = stats_df[stats_df['Property'].str.contains(name.split(' ')[0])]
    if not row.empty:
        p = row['Mann_Whitney_p'].values[0]
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_max = max(max(g) for g in groups if len(g) > 0)
        ax.annotate(f'p={p:.3f} {sig}',
                    xy=(1.5, y_max), ha='center', fontsize=10,
                    fontweight='bold' if p < 0.05 else 'normal',
                    color='black')

    ax.set_ylabel(name, fontsize=11)
    ax.set_title(name.split(' (')[0], fontsize=12, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Log scale for Resistivity specifically to match the 4-panel figure style
axes[0].set_yscale('log')

fig.suptitle(
    'Site C0019J — Physical Property Distributions:\nMTD Intervals vs Stable Matrix',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()

out = os.path.join(OUTPUT_DIR, 'C0019J_MTD_boxplots.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                medianprops=dict(color='black', lw=2),
                widths=0.6,
                showfliers=False) # <--- Add this

/tmp/ipykernel_14573/3269385510.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(groups, labels=labels, patch_artist=True,
/tmp/ipykernel_14573/3269385510.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(groups, labels=labels, patch_artist=True,
/tmp/ipykernel_14573/3269385510.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(groups, labels=labels, patch_artist=True,
/tmp/ipykernel_14573/3269385510.py:80: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropp

## Step 6 — Per-MTD summary

Report the mean physical properties for each individual MTD interval.
This connects the catalog to measurable physical contrasts and allows
direct comparison with published values from the Exp 405 proceedings.

In [ ]:
# Cell 12: Per-MTD physical property summary
print('=== Per-MTD Physical Property Summary ===')
print()
print(f'{"MTD":<8} {"Top":>6} {"Base":>6} {"Thick":>6} {"Res(Ohm)":>10} {"Vp(m/s)":>9} {"Por(v/v)":>9}')
print('-' * 60)

per_mtd = []
for _, row in mtd_catalog.iterrows():
    mtd_id = row['mtd_id']
    top    = row['top_m']
    bottom = row['bottom_m']

    # mean properties within this specific interval
    r_vals = res_c[(res_c['Depth_m'] >= top) & (res_c['Depth_m'] <= bottom)]['Resistivity_Ohm']
    v_vals = vp_c[ (vp_c['Depth_m']  >= top) & (vp_c['Depth_m']  <= bottom)]['Vp_ms']
    p_vals = mad_c[(mad_c['Depth_m'] >= top) & (mad_c['Depth_m'] <= bottom)]['Porosity_vv']

    r_mean = r_vals.mean() if len(r_vals) > 0 else float('nan')
    v_mean = v_vals.mean() if len(v_vals) > 0 else float('nan')
    p_mean = p_vals.mean() if len(p_vals) > 0 else float('nan')

    per_mtd.append({
        'mtd_id':      mtd_id,
        'top_m':       top,
        'bottom_m':    bottom,
        'thickness_m': row['thickness_m'],
        'mean_stability': row['mean_stability'],
        'mean_resistivity_ohm': r_mean,
        'mean_vp_ms':    v_mean,
        'mean_porosity': p_mean,
        'n_res':  len(r_vals),
        'n_vp':   len(v_vals),
        'n_por':  len(p_vals),
    })

    r_str = f'{r_mean:.2f}' if not np.isnan(r_mean) else 'no data'
    v_str = f'{v_mean:.0f}' if not np.isnan(v_mean) else 'no data'
    p_str = f'{p_mean:.3f}' if not np.isnan(p_mean) else 'no data'
    print(f'{mtd_id:<8} {top:>6.1f} {bottom:>6.1f} {row["thickness_m"]:>6.1f} {r_str:>10} {v_str:>9} {p_str:>9}')

per_mtd_df = pd.DataFrame(per_mtd)
out = os.path.join(OUTPUT_DIR, 'C0019J_MTD_properties.csv')
per_mtd_df.to_csv(out, index=False)
print(f'\nSaved: {out}')

=== Per-MTD Physical Property Summary ===

MTD         Top   Base  Thick   Res(Ohm)   Vp(m/s)  Por(v/v)
------------------------------------------------------------
MTD-1      74.0   76.5    2.5    no data   no data   no data
MTD-2     167.5  177.0    9.5       0.90      1679     0.571
MTD-3     253.0  262.5    9.5       1.15      1722     0.542
MTD-4     272.0  291.0   19.0       1.20      1748     0.537
MTD-5     300.5  310.0    9.5       1.16      1738     0.534
MTD-6     406.0  415.5    9.5       1.24      1678     0.529
MTD-7     425.0  453.5   28.5       1.67      1810   no data
MTD-8     606.5  616.0    9.5       1.56      1958     0.480
MTD-9     702.0  711.5    9.5       1.47      1802     0.496
MTD-10    940.0  944.0    4.0    no data   no data   no data

Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_properties.csv


In [ ]:
# Cell 13: Summary statement for dissertation
print('=== Summary for Dissertation Methods/Results ===')
print()

for _, row in stats_df.iterrows():
    prop = row['Property']
    mtd_m  = row['MTD_mean']
    mat_m  = row['Matrix_mean']
    p      = row['Mann_Whitney_p']
    n_mtd  = int(row['MTD_n'])
    n_mat  = int(row['Matrix_n'])
    sig    = 'statistically significant (p<0.05)' if p < 0.05 else 'not statistically significant (p={p:.3f})'
    direction = 'lower' if mtd_m < mat_m else 'higher'

    pct_diff = abs(mtd_m - mat_m) / mat_m * 100

    print(f'{prop}:')
    print(f'  MTD mean = {mtd_m:.3f} (n={n_mtd})')
    print(f'  Stable matrix mean = {mat_m:.3f} (n={n_mat})')
    print(f'  MTD values are {direction} by {pct_diff:.1f}%')
    print(f'  Mann-Whitney U: p = {p:.4f} — {"SIGNIFICANT" if p < 0.05 else "not significant"}')
    print()

=== Summary for Dissertation Methods/Results ===

Resistivity (Ohm-m):
  MTD mean = 1.398 (n=29)
  Stable matrix mean = 20.476 (n=92)
  MTD values are lower by 93.2%
  Mann-Whitney U: p = 0.0304 — SIGNIFICANT

P-wave velocity (m/s):
  MTD mean = 1775.586 (n=29)
  Stable matrix mean = 1849.908 (n=76)
  MTD values are lower by 4.0%
  Mann-Whitney U: p = 0.0017 — SIGNIFICANT

Porosity (v/v):
  MTD mean = 0.525 (n=67)
  Stable matrix mean = 0.511 (n=196)
  MTD values are higher by 2.6%
  Mann-Whitney U: p = 0.0132 — SIGNIFICANT

